In [15]:
import lancedb
import os
import pyarrow as pa
import pyarrow.parquet as pq
import polars as pl
from rich.progress import track
from tqdm import tqdm

In [3]:
def stream_dicts_to_parquet(dict_generator, output_path, schema, chunk_size=10000):
    """
    Streams dictionaries from a generator to a Parquet file.
    """
    with pq.ParquetWriter(output_path, schema) as writer:
        batch_rows = []
        for i, row in enumerate(dict_generator):
            batch_rows.append(row)
            
            # When the chunk is full, convert to RecordBatch and write
            if len(batch_rows) >= chunk_size:
                batch = pa.RecordBatch.from_pylist(batch_rows, schema=schema)
                writer.write_batch(batch)
                batch_rows = []  # Clear for next chunk
        
        # Write any remaining rows
        if batch_rows:
            batch = pa.RecordBatch.from_pylist(batch_rows, schema=schema)
            writer.write_batch(batch)

# --- Example Usage ---

# 1. Define your schema (Mandatory for streaming)
my_schema = pa.schema([
    ('id', pa.int64()),
    ('name', pa.string()),
    ('score', pa.float64())
])

# 2. A mock generator of dicts
def my_data_generator(n=500_000):
    for i in track(range(n)):
        yield {"id": i, "name": f"User_{i}", "score": i * 1.5}

# 3. Run the stream
stream_dicts_to_parquet(
    dict_generator=my_data_generator(),
    output_path="output.parquet",
    schema=my_schema,
    chunk_size=5000  # Adjust based on memory vs. performance needs
)

Output()

In [5]:
splits = {'train': 'plain_text/train-00000-of-00001.parquet', 'test': 'plain_text/test-00000-of-00001.parquet'}
df = pl.read_parquet('hf://datasets/uoft-cs/cifar10/' + splits['train'])

In [12]:
df.select(pl.col("img").struct.field("bytes").alias("img"), "label").write_parquet("cifar10_train.parquet")

In [23]:
from datetime import timedelta
db = lancedb.connect("cache")
tbl = db.create_table("cifar10_train", pl.scan_parquet("cifar10_train.parquet").collect_batches(), mode="overwrite")
tbl.optimize(cleanup_older_than=timedelta(seconds=0), delete_unverified=True)


In [24]:
tbl.take_offsets([1256]).to_list()

[{'img': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00 \x00\x00\x00 \x08\x02\x00\x00\x00\xfc\x18\xed\xa3\x00\x00\x08XIDATx\x9c\x05\xc1\xd9\xae%\xc9U\x00\xd0\xd8C\x0c\x19\x99y\xc6\xaa[C\x0ft{\x00\xdbBj\xf9\x01a\t\x89\x8f@\xfc2\xe2\xc1\xbc \x1b\xa9\xc1\x16\x06\xec\xae\xea{\xab\xeep\xce\xc9\x8c\x8cq\x07k\xc1?\xfd\xf3o\xe6\xd9\x1d\xcfc\x08\xdby\xcf\xda\xd1\x12\xc00\xd7\x1e\x1e\x9f\xd7\xe3\xc1\x0e\x86Rl\xb5\xe2\xe4\x9d\xf7\xb6tAM\xdb\x12\xb5V\xaa\xc3\xbf\xfe\xcb\xef\x06\xb7?\xbf\xda\xbf\xba\xdbi\xadZj\x0f\x0fkQe\xbf\x1f\x1c\r\xc8\n-R\xbc\xa5\xdbc \xe9\xd8\xd1\x80\x91\x9c{M\xde\x1a\xc3d\x8cQJ!\xaa\xd3i7\x0c.\x97J\xc4"\xca\x197\xf9\xfd\xa7\xfb\x8b\xd5\xb35\xc3\xf1x\xdc\x1ff\xa2\xae@v{?\x8fvrCI}\xbde\x1e\x07 t\xd3<h\xd3\xee\x0e\x07\xe7\xddn>\x8c\x93\ti\xa9\xb9\x1b&D\xe5\x07m\x8c\xcd)\xb7*\n\xdb-,w\xc7\xd7R\xd4<\xee\x8f\xfb\n\xcc\xd3\xe4\xb7\xb0\xe6\x92U\xc7-\xdc\x98\xb0\xe5\x9ac\x8d)\xf3\xaf\xbf\xfb\xb9\xb5N\x1be\r\x0elJ\xadh-\xea\xae\x97\xbe\\rH[\xe9\xe0\x1c\x86\x90JJ\xadIH)\xa4x\x81\xeb\